# Partially Observable MDPs (POMDPs)

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/pomdps)

Implements the classic **Tiger problem**: an agent stands before two doors. Behind one is a tiger, behind the other a reward. The agent can LISTEN (noisy observation of where the tiger is) or OPEN a door. We implement the Bayesian belief update and show how listening sharpens the belief before acting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748', 'grid.color': '#2d3748', 'axes.grid': True,
})
np.random.seed(0)

## 1. The Tiger POMDP

States: `TIGER_LEFT`, `TIGER_RIGHT`. Listening yields the correct observation with probability 0.85.

In [ ]:
# Two hidden states
STATES = ['TIGER_LEFT', 'TIGER_RIGHT']
LISTEN_ACCURACY = 0.85  # P(hear tiger-left | tiger really left)

def belief_update(belief, observation):
    """
    Bayesian filter for the LISTEN action.
    belief: [P(TIGER_LEFT), P(TIGER_RIGHT)]
    observation: 'hear_left' or 'hear_right'
    """
    # Observation likelihood O(o | s)
    if observation == 'hear_left':
        likelihood = np.array([LISTEN_ACCURACY, 1 - LISTEN_ACCURACY])
    else:  # hear_right
        likelihood = np.array([1 - LISTEN_ACCURACY, LISTEN_ACCURACY])
    posterior = likelihood * belief
    return posterior / posterior.sum()  # normalize

# Start with a uniform belief — we have no idea where the tiger is
belief = np.array([0.5, 0.5])
print(f"Initial belief: P(left)={belief[0]:.3f}, P(right)={belief[1]:.3f}")

# True state: tiger is on the left. Simulate noisy listens.
true_state = 0  # TIGER_LEFT
beliefs_over_time = [belief.copy()]
for step in range(8):
    # Sample an observation given the true state
    if np.random.rand() < LISTEN_ACCURACY:
        obs = 'hear_left' if true_state == 0 else 'hear_right'
    else:
        obs = 'hear_right' if true_state == 0 else 'hear_left'
    belief = belief_update(belief, obs)
    beliefs_over_time.append(belief.copy())
    print(f"Listen {step+1}: obs={obs:11s} → P(left)={belief[0]:.3f}")

In [ ]:
beliefs_arr = np.array(beliefs_over_time)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(beliefs_arr[:, 0], 'o-', color='#10b981', linewidth=2, label='P(TIGER_LEFT) — true')
ax.plot(beliefs_arr[:, 1], 's-', color='#f43f5e', linewidth=2, label='P(TIGER_RIGHT)')
ax.axhline(0.5, color='white', linestyle=':', alpha=0.4)
ax.set_xlabel('Number of LISTEN actions')
ax.set_ylabel('Belief probability')
ax.set_title('Belief Sharpens with Repeated Noisy Observations', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

print("The belief is a sufficient statistic of the entire action-observation history.")

## 2. Information value: when is it worth listening vs. acting?

Opening the correct door gives +10, the tiger gives −100. Listening costs −1. The optimal policy listens until the belief is confident enough, then opens.

In [ ]:
R_REWARD, R_TIGER, R_LISTEN = 10.0, -100.0, -1.0

def expected_value_open(belief):
    """Best expected value of opening a door given the belief."""
    # Open RIGHT door: bad if tiger is right
    v_open_right = belief[0]*R_REWARD + belief[1]*R_TIGER
    v_open_left  = belief[0]*R_TIGER  + belief[1]*R_REWARD
    return max(v_open_left, v_open_right)

p_left = np.linspace(0, 1, 200)
v_open = [expected_value_open(np.array([p, 1-p])) for p in p_left]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(p_left, v_open, color='#6366f1', linewidth=2, label='Value of opening best door')
ax.axhline(0, color='white', linestyle=':', alpha=0.4, label='Break-even')
ax.fill_between(p_left, v_open, 0, where=(np.array(v_open) < 0), color='#f43f5e', alpha=0.15)
ax.set_xlabel('Belief P(TIGER_LEFT)')
ax.set_ylabel('Expected value of opening')
ax.set_title('Open Only When Confident — Otherwise Keep Listening', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

print("Near belief=0.5, opening has strongly negative value — the agent should pay to LISTEN.")
print("This information-gathering behavior never arises in a fully observable MDP.")

## ✏️ Your turn

**Exercise 1 — Sensor quality.** Repeat the belief-update experiment with `LISTEN_ACCURACY` = 0.6 (noisier sensor) and 0.95 (better sensor). How many listens does it take to push the belief above 0.95 in each case?

In [ ]:
def listens_to_confidence(accuracy, target=0.95, max_listens=50):
    """TODO(you): count LISTEN actions to reach target belief (assume all-correct observations)."""
    pass

In [ ]:
# Assert cell
def listens_ref(accuracy, target=0.95, max_listens=50):
    b = np.array([0.5, 0.5])
    global LISTEN_ACCURACY
    saved = LISTEN_ACCURACY
    LISTEN_ACCURACY = accuracy
    for n in range(1, max_listens+1):
        b = belief_update(b, 'hear_left')
        if b[0] >= target:
            LISTEN_ACCURACY = saved
            return n
    LISTEN_ACCURACY = saved
    return max_listens

for acc in [0.6, 0.85, 0.95]:
    print(f"accuracy={acc}: {listens_ref(acc)} listens to reach 0.95 confidence")
assert listens_ref(0.6) > listens_ref(0.95), "Noisier sensors need more listens"

<details><summary>Solution</summary>

```python
def listens_to_confidence(accuracy, target=0.95, max_listens=50):
    b = np.array([0.5, 0.5])
    global LISTEN_ACCURACY
    LISTEN_ACCURACY = accuracy
    for n in range(1, max_listens+1):
        b = belief_update(b, 'hear_left')   # assume consistently correct obs
        if b[0] >= target:
            return n
    return max_listens
```

A noisier sensor (0.6) needs many more listens to reach the same confidence — each observation carries less information. This is the core POMDP trade-off: the cost of repeated listening vs. the risk of acting under uncertainty.

</details>